# UC2 — a TGF-β time course as one AnnNet object

Four heterogeneous sources — OmniPath signaling, DoRothEA regulation, OmniPath
complexes, Human-GEM metabolism — fused into a single AnnNet object, then
queried, validated, exported and trained on without ever leaving that object.

The substrate is the Saez-lab kidney fibrosis study: human kidney PDGFRb+
mesenchymal cells under TGF-β, profiled by transcriptomics, proteomics,
phosphoproteomics and secretomics across seven timepoints.

**Two mylti-layer (Kivelä) aspects**, both carrying biological meaning:

- `mechanism` ∈ {signaling, regulatory, metabolic} — the process an edge belongs to
- `time` ∈ {0h, 1h, 12h, 24h, 48h, 72h, 96h} — when an entity is *responsive*

A node-layer is therefore e.g. `(prot:COL1A1, signaling, 24h)`, and the time
signal lives in *which layers a node belongs to* rather than in an attribute.

**The four notebooks.** `01` builds the graph and writes `data/uc2.annnet`;
`02`–`04` reload that one file.

## The responsive gate

One gate defines the universe: a gene symbol enters a time layer only where the
study calls it differential at that timepoint. No fold-change floor, so the
early layers stay populated.

In [2]:
from pathlib import Path

import polars as pl
import pyreadr

DATA = Path("data")
ADJP = 0.05
BASELINE = "0h"
TIMES = ("1h", "12h", "24h", "48h", "72h", "96h")

# rna / proteomics / secretomics feature ids are bare symbols; phospho ids look
# like SYMBOL_PEPTIDE___n_SITE, so the symbol is the first underscore field.
measurements = (
    pl.from_pandas(
        pyreadr.read_r(str(DATA / "2024-08-15_diff_results.RData"))["diff_results"][
            ["modality", "feature_id", "logFC", "adj.P.Val", "time"]
        ]
    )
    .rename({"adj.P.Val": "adjp"})
    .filter(pl.col("feature_id").is_not_null())
    .with_columns(symbol=pl.col("feature_id").str.split("_").list.first())
)

# Significant rows aggregated from site level up to (symbol, time), keeping the
# strongest-magnitude logFC. `04` reads this back for the forecasting features.
responsive = (
    measurements.filter(pl.col("adjp") < ADJP)
    .sort(pl.col("logFC").abs(), descending=True)
    .group_by("symbol", "time")
    .agg(best_logFC=pl.col("logFC").first())
)
responsive.write_parquet(DATA / "responsive.parquet")

measured_symbols = set(measurements["symbol"])
responsive_at = {
    t: set(group["symbol"]) for t, group in responsive.to_pandas().groupby("time", observed=True)
}

print(f"measured universe: {len(measured_symbols):,} symbols")
print("responsive per timepoint:", {t: len(responsive_at[t]) for t in TIMES})

measured universe: 14,966 symbols
responsive per timepoint: {'1h': 529, '12h': 2762, '24h': 3297, '48h': 4464, '72h': 5078, '96h': 7129}


## The aspect grid

`set_aspects` declares the two aspects and their elementary layers. The layer
space is their product, so every node placement below names a coordinate.

In [3]:
import annnet as an
import omnipath as op
import omnipath_client as oc

SNAPSHOT = DATA / "uc2.annnet"

G = an.AnnNet(directed=True)
G.history.enable(True)
G.history.snapshot("init")
G.layers.set_aspects(
    ["mechanism", "time"],
    {"mechanism": ["signaling", "regulatory", "metabolic"], "time": [BASELINE, *TIMES]},
)

# or directly: G = an.AnnNet(directed=True, aspects= {"mechanism": ["signaling", "regulatory", "metabolic"], "time": [BASELINE, *TIMES]})

print("aspects:", G.layers.list_aspects())
print("layers :", G.layers.list_layers())

aspects: ('mechanism', 'time')
layers : {'mechanism': ['metabolic', 'regulatory', 'signaling'], 'time': ['0h', '12h', '1h', '24h', '48h', '72h', '96h']}


## Responsive layers — signaling and regulatory

`omnipath.interactions.OmniPath` fetches the prior-knowledge signaling network
and `omnipath_client.to_annnet` turns it into a graph; DoRothEA (confidence A
and B) supplies the regulatory one. For each mechanism,
`add_time_layers` walks the six timepoints and keeps only the edges whose
endpoints are both responsive then — a pure 0-hop restriction on the gate.
The same protein therefore appears once per timepoint it responds in.

In [4]:
pkn = oc.to_annnet(
    op.interactions.OmniPath.get(genesymbols=True),
    source_col="source_genesymbol",
    target_col="target_genesymbol",
    edge_attr_cols=["is_stimulation", "is_inhibition"],
)


def sign(stimulation, inhibition):
    """Signed weight from the direction flags; 0.0 when both or neither is set."""
    return 1.0 if stimulation and not inhibition else -1.0 if inhibition and not stimulation else 0.0


signaling_pairs = [
    (r.source, r.target, sign(r.is_stimulation, r.is_inhibition))
    for r in pkn.views.edges().to_pandas().itertuples(index=False)
]
dorothea = pl.read_csv(DATA / "dorothea.tsv", separator="\t", infer_schema_length=5000)
regulatory_pairs = [
    (r["source_genesymbol"], r["target_genesymbol"], sign(r["is_stimulation"], r["is_inhibition"]))
    for r in dorothea.iter_rows(named=True)
]
print(f"prior knowledge: {len(signaling_pairs):,} signaling, {len(regulatory_pairs):,} regulatory")

/home/l1boll/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[timing] fetch/receive df:     2.324s
[timing] column resolution:    0.0160s
         source='source_genesymbol'  target='target_genesymbol'  directed='is_directed'
         edge_attr_cols (2): ['is_stimulation', 'is_inhibition']
[timing] AnnNet init:          0.002s  (pre-sized n=85217 e=85217)
[timing] to_rows setup:        0.000s  (85217 rows, streaming=True)
[timing] bulk list build:      0.333s  (85217 edges)
[timing] add_edges_bulk:       0.729s
         vertices=8791  edges=85217
prior knowledge: 85,217 signaling, 15,267 regulatory


In [5]:
def add_time_layers(G, mechanism, pairs, prefix, kind):
    """Place responsive-gated supra-nodes and edges for one mechanism, per timepoint.

    Returns the {(symbol, time)} placed, which the coupling step consumes.
    """
    placed = set()
    for t in TIMES:
        active = responsive_at[t]
        kept = [(a, b, w) for a, b, w in pairs if a in active and b in active]
        symbols = {v for a, b, _ in kept for v in (a, b)}
        G.add_nodes(
            [{"node_id": f"{prefix}{v}", "gene_symbol": v, "kind": kind} for v in sorted(symbols)],
            layer=(mechanism, t),
        )
        G.add_edges(
            [
                {
                    "source": (f"{prefix}{a}", (mechanism, t)),
                    "target": (f"{prefix}{b}", (mechanism, t)),
                    "weight": w,
                    "edge_kind": mechanism,
                }
                for a, b, w in kept
            ],
            default_edge_directed=True,
        )
        placed |= {(v, t) for v in symbols}
    return placed


signaling_placed = add_time_layers(G, "signaling", signaling_pairs, "prot:", "protein")
regulatory_placed = add_time_layers(G, "regulatory", regulatory_pairs, "gene:", "gene")
G.history.snapshot("after_response")

print(f"{'time':>5} {'signaling |V|':>14} {'regulatory |V|':>15}")
for t in TIMES:
    print(
        f"{t:>5} {len(G.layers.layer_node_set(('signaling', t))):>14,}"
        f" {len(G.layers.layer_node_set(('regulatory', t))):>15,}"
    )

 time  signaling |V|  regulatory |V|
   1h            107             116
  12h            815             513
  24h          1,131             750
  48h          1,651           1,063
  72h          1,955           1,221
  96h          2,984           1,840


## Interlayer coupling

**Time coupling** links the same entity at consecutive timepoints,
`(v, mech, tᵢ) → (v, mech, tᵢ₊₁)`, wherever it is responsive in both — genuine
diagonal structure in the `time` aspect. **Translation coupling** links a
responsive gene to its protein within one timepoint, bridging the two
mechanisms. Both are ordinary edges with layer-qualified endpoints; `03`
checks every one of them.

In [6]:
def at(placed, t):
    """Symbols placed in layer `t`."""
    return {v for v, tt in placed if tt == t}


time_edges = [
    {
        "edge_id": f"time:{mechanism}:{v}:{a}->{b}",
        "edge_kind": "coupling_time",
        "weight": 1.0,
        "source": (f"{prefix}{v}", (mechanism, a)),
        "target": (f"{prefix}{v}", (mechanism, b)),
    }
    for mechanism, prefix, placed in [
        ("signaling", "prot:", signaling_placed),
        ("regulatory", "gene:", regulatory_placed),
    ]
    for a, b in zip(TIMES, TIMES[1:])
    for v in sorted(at(placed, a) & at(placed, b))
]

translation_edges = [
    {
        "edge_id": f"trans:{v}:{t}",
        "edge_kind": "coupling_translation",
        "weight": 1.0,
        "source": (f"gene:{v}", ("regulatory", t)),
        "target": (f"prot:{v}", ("signaling", t)),
    }
    for t in TIMES
    for v in sorted(at(regulatory_placed, t) & at(signaling_placed, t))
]

G.add_edges(time_edges, default_edge_directed=True)
G.add_edges(translation_edges, default_edge_directed=True)
G.history.snapshot("after_coupling")
print(f"time-coupling edges       : {len(time_edges):,}")
print(f"translation-coupling edges: {len(translation_edges):,}")

time-coupling edges       : 7,660
translation-coupling edges: 3,322


## Baseline scaffold — two kinds of hyperedge

Complexes and metabolism are time-invariant, so they live in the `0h` layer.
Each OmniPath complex becomes **one undirected hyperedge** over its subunits —
a single incidence-matrix column, not a clique of pairwise edges. One
`from_sbml` call reads Human-GEM: reactions become **signed directed
hyperedges** carrying stoichiometry, and compartments become slices.

`from_sbml` adds metabolite and boundary nodes without a `kind`, so they are
typed here for the `to_pyg` export in `04`.

In [10]:
complexes = pl.read_csv(DATA / "omnipath_complexes.tsv", separator="\t", infer_schema_length=5000)
complex_specs, members = [], set()
for row in complexes.iter_rows(named=True):
    subunits = (row["components_genesymbols"] or "").split("_")
    if subunits == [""] or not set(subunits) <= measured_symbols:
        continue
    members.update(f"prot:{s}" for s in subunits)
    complex_specs.append(
        {
            "edge_id": f'cpx:{row["name"]}',
            "edge_kind": "complex",
            "weight": 1.0,
            "members": [(f"prot:{s}", ("signaling", BASELINE)) for s in subunits],
        }
    )

G.add_nodes(
    [{"node_id": v, "kind": "protein", "gene_symbol": v.removeprefix("prot:")} for v in sorted(members)],
    layer=("signaling", BASELINE),
)
G.add_edges(complex_specs, layer=("signaling", BASELINE))
G.history.snapshot("after_complex")

an.from_sbml(
    str(DATA / "Human-GEM.xml"),
    graph=G,
    slice="metabolic",
    layer=("metabolic", BASELINE),
    preserve_stoichiometry=True,
)
untyped = G.views.nodes().to_pandas()
untyped = untyped[untyped["kind"].isna()]["node_id"]
G.attrs.set_node_attrs_bulk(
    {v: {"kind": "boundary" if v.startswith("__") else "metabolite"} for v in untyped}
)
G.history.snapshot("after_metabolic")

# Complex names are not unique in OmniPath, so same-named entries share an edge_id.
n_complex = sum(str(e).startswith("cpx:") for e in G.hyperedge_definitions)
print(f"complex hyperedges  : {n_complex:,} over {len(members):,} proteins")
print(f"reaction hyperedges : {len(G.hyperedge_definitions) - n_complex:,} (Human-GEM, signed stoichiometry)")

complex hyperedges  : 3,446 over 8,091 proteins
reaction hyperedges : 12,971 (Human-GEM, signed stoichiometry)


## Organelle slices

A slice is a **membership overlay** on the shared graph: it duplicates no
topology, where the usual alternative keeps one graph object per compartment.
Compartments are time-invariant, so a protein joins its organelle regardless of
when it responds. `03` and `04` read the membership back out of the graph.

In [12]:
ORGANELLES = {
    "Mitochondria": "mitochondria",
    "Nucleus": "nucleus",
    "Nucleoplasm": "nucleus",
    "Nucleoli": "nucleus",
    "Nuclear membrane": "nucleus",
    "Endoplasmic reticulum": "er",
    "Golgi apparatus": "golgi",
    "Lysosome": "lysosome",
    "Cytosol": "cytosol",
    "Cytoplasm": "cytosol",
    "Plasma membrane": "plasma_membrane",
    "Cell Junctions": "plasma_membrane",
    "Peroxisome": "peroxisome",
    "Vesicles": "vesicles",
}

hpa = pl.read_csv(DATA / "proteinatlas.tsv", separator="\t", infer_schema_length=10000)
location = dict(zip(hpa["Gene"].to_list(), hpa["Subcellular main location"].to_list()))


def organelle_of(symbol):
    """Organelle slice name for a gene symbol; cytosol by default."""
    text = location.get(symbol) or ""
    return next((name for key, name in ORGANELLES.items() if key.lower() in text.lower()), "cytosol")


proteins = G.views.nodes().to_pandas()
proteins = proteins[proteins.node_id.str.startswith("prot:")]
organelle = {r.node_id: organelle_of(r.gene_symbol) for r in proteins.itertuples()}

for name in sorted(set(organelle.values())):
    vids = {v for v, o in organelle.items() if o == name}
    G.slices.add(name, role="organelle")
    for v in sorted(vids):
        G.slices.add_node_to_slice(name, v)
    G.slices.add_edges(
        name,
        [e for e, (s, t, _) in G.edge_definitions.items() if s[0] in vids and t[0] in vids],
    )

G.history.snapshot("after_organelle_slices")

for name in sorted(set(organelle.values())):
    print(f"{name:>16}: {len(G.slices.nodes(name)):>5,} nodes, {len(G.slices.edges(name)):>5,} edges")

         cytosol: 3,469 vertices, 5,370 edges
              er:   277 vertices,   218 edges
           golgi:   369 vertices,   294 edges
        lysosome:    17 vertices,     7 edges
    mitochondria:   609 vertices,   229 edges
         nucleus: 3,251 vertices, 6,451 edges
      peroxisome:    11 vertices,     7 edges
 plasma_membrane:   513 vertices, 1,421 edges
        vesicles:   490 vertices,   350 edges


## Write the snapshot

One `.annnet` file holds both aspects, both hyperedge families, the slices, the
coupling, every attribute table and the build history.

In [16]:
G.write(str(SNAPSHOT), overwrite=True)
print(f"{SNAPSHOT} ({SNAPSHOT.stat().st_size / 1e6:.1f} MB)")
print(f"|V| = {G.ncount():,}   |E| = {G.ecount():,}")
print(f"binary edges {len(G.edge_definitions):,} | hyperedges {len(G.hyperedge_definitions):,}"
      f" | slices {len(G.slices.list()):,}")

data/uc2.annnet (5.1 MB)
|V| = 19,587   |E| = 63,891
binary edges 47,474 | hyperedges 16,417 | slices 20
